In [1]:
pip install chromadb sentence-transformers python-dotenv

Defaulting to user installation because normal site-packages is not writeable
  Using cached sentence_transformers-6.1.0-py3-none-any.whl.metadata (20 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
Using cached sentence_transformers-6.1.0-py3-none-any.whl (740 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   -

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
# Verification Cell for Step 2 Dependencies
import chromadb
from chromadb.utils import embedding_functions
import dotenv

print("Checking dependencies...")
print(f"ChromaDB Version: {chromadb.__version__}")

# Initialize local embedding function test
test_emb = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
sample_vector = test_emb(["Testing embedding generator"])

print("Embedding model loaded successfully!")
print(f"Sample Embedding Vector Dimension: {len(sample_vector[0])}")
print("All Step 2 modules are installed and ready!")

Checking dependencies...
ChromaDB Version: 1.5.9


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\reshu\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\reshu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!
Sample Embedding Vector Dimension: 384
All Step 2 modules are installed and ready!


In [3]:
import os
import chromadb
from chromadb.utils import embedding_functions

# --- 1. Hard Disk Par Persistent Vector Database Client Banana ---
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# --- 2. Local Sentence-Transformer Embedding Model Setup ---
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# --- 3. Collection (Table) Create ya Retrieve Karna ---
collection = chroma_client.get_or_create_collection(
    name="procurement_policies",
    embedding_function=embedding_fn
)

# --- 4. Markdown Policy File Ko Read Karna ---
policy_file_path = "data/procurement_policy.md"

with open(policy_file_path, "r", encoding="utf-8") as f:
    raw_policy_text = f.read()

# --- 5. Document Ko Chunks (Paragraphs) Me Split Karna ---
# Hum text ko double line breaks (\n\n) se split kar rahe hain
policy_chunks = [chunk.strip() for chunk in raw_policy_text.split("\n\n") if chunk.strip()]

# Har chunk ke liye unique IDs generate karna (e.g., rule_1, rule_2, etc.)
chunk_ids = [f"rule_{i+1}" for i in range(len(policy_chunks))]

# --- 6. Chunks Ko Vector Database Me Index/Save Karna ---
# Duplicate entry avoid karne ke liye count check kar rahe hain
if collection.count() == 0:
    collection.add(
        documents=policy_chunks,
        ids=chunk_ids,
        metadatas=[{"source": "procurement_policy.md"} for _ in policy_chunks]
    )
    print(f"Successfully indexed {collection.count()} policy chunks into ChromaDB!")
else:
    print(f"Collection already contains {collection.count()} indexed chunks. Skipping re-indexing.")

# --- 7. VERIFICATION QUERY: Vector DB Search Test ---
test_query = "What is the policy for invoice total exceeding PO limit?"

search_results = collection.query(
    query_texts=[test_query],
    n_results=2  # Top 2 most relevant chunks mangwa rahe hain
)

print("\n--- Verification: Retrieved Policy Context ---")
for idx, doc in enumerate(search_results["documents"][0], start=1):
    print(f"Result {idx}:\n{doc}\n")

Successfully indexed 5 policy chunks into ChromaDB!

--- Verification: Retrieved Policy Context ---
Result 1:
## Rule POL_VAR_01: Purchase Order Variance Limit
- Any invoice where the total requested amount exceeds the associated Purchase Order (PO) limit by more than 5.0% must be flagged for manual supervisor review.
- Variances under 5.0% or up to $50 absolute difference are auto-approved for processing.

Result 2:
## Rule POL_TAX_02: Mandatory Tax ID Validation
- Every invoice with a total net value exceeding $1,000.00 USD must explicitly include a valid Vendor Tax ID or VAT Registration Number.
- Invoices missing Tax IDs above this threshold must be placed on payment block.



In [4]:
with open("data/procurement_policy.md", "r", encoding="utf-8") as f:
    print(f.read()[:500])  # Prints first 500 characters of your policy file

# Enterprise Procurement Policy Guidelines & Audit Framework

## Rule POL_VAR_01: Purchase Order Variance Limit
- Any invoice where the total requested amount exceeds the associated Purchase Order (PO) limit by more than 5.0% must be flagged for manual supervisor review.
- Variances under 5.0% or up to $50 absolute difference are auto-approved for processing.

## Rule POL_TAX_02: Mandatory Tax ID Validation
- Every invoice with a total net value exceeding $1,000.00 USD must explicitly include a 
